In [32]:
from Constants import path_strings
from dotenv import load_dotenv
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from sqlalchemy import create_engine, Engine, text, URL
import os
import src.transformation as trf
import src.load as ld
import urllib.parse

load_dotenv("../.env")

pio.renderers.default = "iframe"

# Bring data from .env file
user = os.getenv("USER")
password = os.getenv("PASSWORD")
host = os.getenv("HOST")
port = os.getenv("PORT")
db_name = os.getenv("DB_NAME")

# 1. Create the URL object (Safe way)
connection_url = URL.create(
    drivername="postgresql",
    username=user,
    password=password, # No need to manually quote_plus here!
    host=host,
    port=port,
    database=db_name
)

# 2. Establish connection
engine = create_engine(connection_url)


# Profile Report

## Pre transformation profile

### Characterization and Quality

In [33]:
raw_data = trf.load_bronze(path_strings.raw_main_path)
print(trf.get_profile(raw_data, "Pre_transformation"))
print(raw_data.info())
print(raw_data.describe())

SyntaxError: invalid syntax (2710786978.py, line 4)

## Post transformation profile

### Characterization and Quality - Considered only the Fact Emissions (National) to ensure normalization

In [35]:
national_data = ld.load_silver(path_strings.silver_national_path)
print(trf.get_profile(national_data, "Post_transformation"))
print(national_data.info())
print(national_data.describe())

{'stage': 'Post_transformation', 'rows': 38407, 'cols': 79, 'null_sum': np.int64(1416855), 'memory_mb': np.float64(27.020797729492188), 'dtypes': {dtype('float64'): 75, dtype('O'): 2, dtype('int32'): 1, dtype('int64'): 1}}
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38407 entries, 0 to 38406
Data columns (total 79 columns):
 #   Column                                         Non-Null Count  Dtype  
---  ------                                         --------------  -----  
 0   country                                        38407 non-null  object 
 1   year                                           38407 non-null  int32  
 2   iso_code                                       38407 non-null  object 
 3   population_(people)                            38407 non-null  int64  
 4   gdp_($)                                        15199 non-null  float64
 5   cement_co2_(Mt)                                21823 non-null  float64
 6   cement_co2_per_capita_(t/person)               21823 no

### Graphics report

In [ ]:
# Base for the Graphics
snapshot_pre_transformation = trf.get_profile(raw_data, "Pre_transformation")

snapshot_post_transformation = trf.get_profile(national_data, "Post_transformation")

comparing_table = pd.DataFrame([snapshot_pre_transformation, snapshot_post_transformation])
print(comparing_table)

### G1. Memory Optimization (The Efficiency Gain)

In [ ]:
g1 = px.bar(comparing_table, x='stage', y='memory_mb', color='stage',
             title='1. Memory Footprint Reduction',
             labels={'Memory_MB': 'Memory Usage (MB)'},
             text_auto='.2f', color_discrete_sequence=['#ef553b', '#00cc96'])
g1.show()

### G2. Null Values (Quality gain)

In [ ]:
g2 = px.bar(comparing_table, x='stage', y='null_sum', color='stage',
             title='2. Total Null Values: Raw vs. Cleaned',
             text_auto=True, color_discrete_sequence=['#ef553b', '#00cc96'])
g2.show()

### G3. Schema Evolution: Column Count

In [ ]:
g3 = px.bar(comparing_table, x='stage', y='rows', color='stage',
             title='3. Schema Simplification (Row Count)',
             text_auto=True, color_discrete_sequence=['#636efa', '#ab63fa'])
g3.show()

### G4. Type Casting: Objects vs. Numerics

In [ ]:
# 1. Calculate KB per row for both stages
comparing_table['KB_per_Row'] = (comparing_table['memory_mb'] * 1024) / comparing_table['rows']

# 2. Plot the efficiency gain
g4 = px.bar(comparing_table, x='stage', y='KB_per_Row', color='stage',
             title='4. Data Density: Memory per Row (Efficiency)',
             labels={'stage': 'Pipeline Stage', 'KB_per_Row': 'Kilobytes per Row'},
             text_auto='.4f',
             color_discrete_sequence=['#ef553b', '#00cc96'])

# Add a horizontal line or annotation showing the improvement
g4.update_layout(showlegend=False)
g4.show()

### G5. Data Range Consistency (Boxplot)

In [37]:
# Comparing the distribution of the same column before and after
# (Assuming 'co2' exists in both)
g5 = go.Figure()
g5.add_trace(go.Box(y=raw_data['co2'], name='Bronze CO2', marker_color='#ef553b'))
g5.add_trace(go.Box(y=national_data['co2_(Mt)'], name='Silver CO2 (Standardized)', marker_color='#00cc96'))
g5.update_layout(title='5. Distribution & Outlier Management (CO2 Units)')
g5.show()

NameError: name 'go' is not defined

# GOLD LAYER - Business Metrics Analysis

## 1. Global & Regional Trends (The "bigger picture")

### 1.1 The great decoupling: Comparing the growth rate of GDP (Gross Domestic Product) over Total emissions for Brazil in last 30 Years (Considering Greenhouse effects)

In [26]:
query_1 = "SELECT country, year, iso_code, gdp_usd / 1e9 AS gdp_bilions, population_people, total_ghg_mt, total_ghg_mt / NULLIF(gdp_usd / 1e9, 0) AS ghg_intensity FROM co2_project.fact_non_co2_ghg WHERE country = 'Brazil' AND year >= 1996;"
df_1 = ld.run_query(query_1, engine)
print(df_1)

# 2. Plot the "Efficiency Trend"
fig = px.line(df_1, x='year', y='ghg_intensity',
              title='<b>Brazil: Emissions Intensity (Million tonnes of CO2-equivalent per Billion USD GDP)</b>',
              labels={'ghg_intensity': 'Intensity (Lower is Better)'},
              template='plotly_white')

# Add a marker for visual clarity
fig.update_traces(line_color='seagreen')
fig.show()


   country  year iso_code  gdp_bilions  population_people  total_ghg_mt  \
0   Brazil  1996      BRA  1520.364860          164202549      2586.919   
1   Brazil  1997      BRA  1591.956040          166661662      2696.838   
2   Brazil  1998      BRA  1617.675774          169159651      2929.596   
3   Brazil  1999      BRA  1645.887982          171641544      2965.704   
4   Brazil  2000      BRA  1739.728034          174018278      2823.803   
5   Brazil  2001      BRA  1785.513006          176301201      2783.375   
6   Brazil  2002      BRA  1861.742324          178503485      2981.136   
7   Brazil  2003      BRA  1904.932706          180622688      3846.776   
8   Brazil  2004      BRA  2038.447931          182675144      3313.758   
9   Brazil  2005      BRA  2129.214530          184688101      3057.048   
10  Brazil  2006      BRA  2240.805319          186653099      2819.171   
11  Brazil  2007      BRA  2405.626902          188552311      2745.828   
12  Brazil  2008      BRA

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


Result: Segundo a Análise do gráfico, o Aumento no PIB ...

### 1.2 Carbon Leakage: Comparing Co2 Emissions against CO2 consumption for High-income countries since 1990

In [24]:
query_2 = """
SELECT
	e.country,
	e.year,
	e.co2_mt,
	c.consumption_co2_mt,
	-- The Leakage Calculation
	(c.consumption_co2_mt - e.co2_mt) AS carbon_gap
FROM co2_project.agg_emissions AS e
JOIN co2_project.agg_consumption AS c
ON e.country = c.country
AND e.year = c.year
WHERE e.country = 'High-income countries'
AND e.year >= 1990
"""
df_2 = ld.run_query(query_2, engine)
print(df_2)

# 2. Plot the "Carbon Leakage"
fig = px.line(df_2, x='year', y='carbon_gap',
              title='<b>High-income Countries: Difference between Consumption and Production of Co2 (Million tonnes of CO2)</b>',
              template='plotly_white',
              labels={'carbon_gap': 'Carbon gap (Mt)'})
# Adding y = 0 for context
fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Break-even point")
fig.update_traces(fill='tozeroy') 
fig.update_traces(line_color='seagreen')
fig.show()


                  country  year     co2_mt  consumption_co2_mt  carbon_gap
0   High-income countries  1990  15006.094           15283.517     277.423
1   High-income countries  1991  15335.967           15227.156    -108.811
2   High-income countries  1992  14510.799           14729.639     218.840
3   High-income countries  1993  14538.071           14788.983     250.912
4   High-income countries  1994  14511.072           14986.450     475.378
5   High-income countries  1995  14628.970           15171.210     542.240
6   High-income countries  1996  14979.279           15451.955     472.676
7   High-income countries  1997  14900.450           15461.409     560.959
8   High-income countries  1998  14859.571           15452.991     593.420
9   High-income countries  1999  15028.272           15639.714     611.442
10  High-income countries  2000  15461.338           16294.402     833.064
11  High-income countries  2001  15468.816           16283.549     814.733
12  High-income countries

Result: Como é possível de se ver no gráfico, os "high-income countries" são grandes importadores de Carbono. Consumindo mais do que produzem (principalmente próximo ao ano 2000, dada a acelerada globalização), essencialmente "transferindo" a fonte da sua poluição parahubs de manufatura, como Índia e China.

## 2. South America insights

### 2.1 The Amazon Effect: See how the land use in Brazil impact in the total greenhouse gas emissions

In [28]:
query_3 = """
SELECT 
	es.country,
	es.year,
	es.land_use_change_co2_mt,
	c.total_ghg_mt,
	(es.land_use_change_co2_mt / c.total_ghg_mt) * 100 AS land_use_over_ghg_prct
FROM co2_project.fact_emission_sources AS es
JOIN co2_project.fact_non_co2_ghg AS c
ON es.country = c.country
AND es.year = c.year
WHERE es.country = 'Brazil'
AND es.year >= 1990
"""
df_3 = ld.run_query(query_3, engine)
print(df_3)

# 3. Plot the "Amazon impact"
fig = px.line(df_3, x='year', y='land_use_over_ghg_prct',
              title='<b>Brazil: Contribution of land-use in greenhouse emissions (%)</b>',
              template='plotly_white',
              labels={'land_use_over_ghg_prct': 'Land use / ghg (%)'})
# Optional: Highlight the period of major deforestation control
fig.add_vrect(x0=2004, x1=2012, 
              fillcolor="green", opacity=0.1, 
              layer="below", line_width=0,
              annotation_text="Major Deforestation Control")
fig.update_traces(line_color='seagreen')
fig.show()

   country  year  land_use_change_co2_mt  total_ghg_mt  land_use_over_ghg_prct
0   Brazil  1990                1538.253      2105.299               73.065774
1   Brazil  1991                1552.012      2189.194               70.894220
2   Brazil  1992                1623.442      2267.928               71.582608
3   Brazil  1993                1567.832      2282.487               68.689635
4   Brazil  1994                1957.235      2672.895               73.225286
5   Brazil  1995                1886.966      2645.838               71.318274
6   Brazil  1996                1797.620      2586.919               69.488840
7   Brazil  1997                1901.934      2696.838               70.524592
8   Brazil  1998                2142.159      2929.596               73.121311
9   Brazil  1999                2175.559      2965.704               73.357253
10  Brazil  2000                1949.836      2823.803               69.050001
11  Brazil  2001                1852.799      2783.3

Result: Como se pode ver no gráfico, graças a Amazonia, a participação do uso da Terra no Brazil (desmatamento), tem um grande impacto na emissao de gases efeito estufa. Essa participação cai de 2004-2012 a medida que cai o desmatamento. E volta a subir próximo a 2012.

### 2.2 South America Leaderboard of emissions

In [ ]:
query_4 = """
SELECT 
	e.country,
	SUM(e.co2_per_capita_t_per_person + c.ghg_per_capita_t_per_person) AS cumulative_intensity
FROM co2_project.fact_emissions AS e
JOIN co2_project.fact_non_co2_ghg AS c
ON e.country = c.country
AND e.year = c.year
WHERE e.country IN ('Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay', 'Venezuela')
AND e.year >= 2001
GROUP BY e.country
ORDER BY cumulative_intensity ASC
"""
df_4 = ld.run_query(query_4, engine)
print(df_4)

# 4. Plot of the Leaderboard
fig = px.bar(
    df_4, x='cumulative_intensity', y='country',
    title='<b>South America: Ranking of biggest emissions of Co2 per capita (tonnes/person)</b>',
    color='cumulative_intensity',
    labels={'cumulative_intensity': 'Tonnes per Person (CO2 + GHG)'},
    template='plotly_white'
)
# Add an annotation to explain the Guyana result
fig.add_annotation(x=df_4.cumulative_intensity.max(), y='Guyana',
            text="Oil Boom + Small Population",
            showarrow=True, arrowhead=1)
fig.show()

Result: Como esperado Brasil, por ser o maior em população, está num dos primeiros do ranking, perdendo apenas para Guyana. Como o país possui uma pequena população (menos de 1M habitantes) e com a descoberta de uma zona de petróleo na costa, as emissões_per_capita dispararam.

## 3. Sector & Source Deep Dives

### 3.1 The Hidden Polluters: Identify top 5 countries where Nitrous Oxide (N2O) represents the highest percentage of their total GHG footprint in the last 20 years

In [ ]:
query_5 = """
SELECT
    country,
    (SUM(nitrous_oxide_mt) / SUM(NULLIF(total_ghg_mt, 0))) * 100 AS hidden_impact
FROM co2_project.fact_non_co2_ghg
WHERE year >= 2001
  AND total_ghg_mt > 0 -- Security filter
GROUP BY country
ORDER BY hidden_impact DESC
LIMIT 5;
"""
df_5 = ld.run_query(query_5, engine)
print(df_5)

# 5. Plot 
fig = px.bar(
    df_5, x='country', y='hidden_impact',
    title='<b>Top 5 Hidden Polluters: Nitrous Oxide Intensity (2001-2021)</b>',
    labels={'hidden_impact': 'N2O Share (% of Total GHG)', 'country': 'Country'},
    color='hidden_impact',
    color_continuous_scale='OrRd', # Orange to Red scale
    text_auto='.2f', # This puts the exact percentage on top of the bars
    template='plotly_white'
)

# 3. Clean up the layout
fig.update_layout(showlegend=False)
fig.show()

### 3.2 Identifying "Coal" to "Gas" transition in a certain region

In [ ]:
query_6 = """
SELECT 
	year,
    coal_co2_mt,
	gas_co2_mt
FROM co2_project.fact_emission_sources
WHERE country = 'Chile' AND year >= 1990;
"""
df_6 = ld.run_query(query_6, engine)
print(df_6)

# 6. Plotting the transition
fig = px.line(df_6,
              x="year", 
              y=["coal_co2_mt", "gas_co2_mt"], 
              title="<b>Chile: The Coal to Gas Transition (2000-2021)</b>",
              labels={"value": "CO2 Emissions (Mt)", "variable": "Fuel Source"},
              color_discrete_map={"coal_co2": "#333333", "gas_co2": "#3366CC"}, # Grey for coal, Blue for gas
              template="plotly_white")

fig.show()

# 4. Climate and Economy impact

### 4.1 Visualization of the effect of the cumulative CO2 emission into Temperature change in degres Celsius

In [ ]:
query_7 = """
SELECT 
	e.country,
	e.year,
    e.cumulative_co2_mt,
	t.temperature_change_from_co2_degrees_c
FROM co2_project.fact_emissions AS e
JOIN co2_project.fact_climate_impact AS t
ON e.country = t.country
AND e.year = t.year
WHERE e.country = 'Brazil' 
AND e.year >= 2000
AND e.cumulative_co2_mt IS NOT NULL
ORDER BY country, year;
"""
df_7 = ld.run_query(query_7, engine)
print(df_7)
fig = px.scatter(
    df_7, 
    x="cumulative_co2_mt", 
    y="temperature_change_from_co2_degrees_c", 
    color="country",
    hover_name="year",
    title="<b>Climate Sensitivity: The Impact of Cumulative CO2</b>",
    labels={"cumulative_co2_mt": "Cumulative CO2 (Mt)", 
             "temperature_change_from_co2_degrees_c": "Temperature Anomaly (°C)"},
    template="plotly_white"
)

# Make the dots slightly transparent to see the trendlines clearly
fig.update_traces(marker=dict(opacity=0.6, size=8))
fig.show()